# Module 3: GGUF Format Inspection

This notebook uses our `GGUFReader` to explore how llama.cpp stores quantized models.

**What you'll learn:**
- How GGUF binary format is structured
- How to read tensor metadata without loading all weights
- How Q8_0 and Q4_K block dequantization works
- Why GGUF file size is much smaller than FP32 equivalent

**Read first:** `docs/gguf_format.md`

**Time:** ~30 minutes

**You need:** Download a GGUF file first:
```bash
pip install huggingface_hub
huggingface-cli download bartowski/Llama-3.2-1B-Instruct-GGUF \
  --include 'Llama-3.2-1B-Instruct-Q4_K_M.gguf' \
  --local-dir ./models/
```

In [ ]:
import sys, os
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import numpy as np
import struct

from src.gguf.reader import GGUFReader
from src.gguf.k_quants import dequantize_q8_0, dequantize_q4_k

# Set your GGUF file path here
GGUF_PATH = '../models/Llama-3.2-1B-Instruct-Q4_K_M.gguf'

if not os.path.exists(GGUF_PATH):
    print('GGUF file not found! Download it with:')
    print('  huggingface-cli download bartowski/Llama-3.2-1B-Instruct-GGUF')
    print('    --include "Llama-3.2-1B-Instruct-Q4_K_M.gguf" --local-dir ./models/')
else:
    print(f'Found GGUF file: {GGUF_PATH}')
    file_size_mb = os.path.getsize(GGUF_PATH) / (1024**2)
    print(f'File size: {file_size_mb:.1f} MB')

In [ ]:
# Part 1: Read GGUF file and inspect structure
reader = GGUFReader(GGUF_PATH)
print(reader.summary())

In [ ]:
# Part 2: Explore metadata
print('All metadata key-value pairs:')
print('-' * 60)
for key, value in reader.metadata.items():
    if isinstance(value, list):
        print(f'{key}: [list of {len(value)} items]')
    elif isinstance(value, str) and len(value) > 50:
        print(f'{key}: "{value[:50]}..."')
    else:
        print(f'{key}: {value}')

In [ ]:
# Part 3: Tensor inventory
print(f'Total tensors: {len(reader.tensor_infos)}')
print()

# Count by quantization type
from collections import Counter
type_counts = Counter(t.dtype_name for t in reader.tensor_infos)
print('Tensors by quantization type:')
for dtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f'  {dtype}: {count} tensors')

print()
total_bytes = sum(t.size_bytes for t in reader.tensor_infos)
fp32_equivalent = sum(t.n_elements * 4 for t in reader.tensor_infos)
print(f'Total GGUF data: {total_bytes/1024/1024:.1f} MB')
print(f'FP32 equivalent: {fp32_equivalent/1024/1024:.1f} MB')
print(f'Compression ratio: {fp32_equivalent/total_bytes:.2f}x')

In [ ]:
# Part 4: Load and inspect a specific tensor
tensor_name = 'blk.0.attn_q.weight'
tensor_info = next(t for t in reader.tensor_infos if t.name == tensor_name)

print(f'Tensor: {tensor_name}')
print(f'Shape: {tensor_info.shape}')
print(f'Type: {tensor_info.dtype_name}')
print(f'Raw size: {tensor_info.size_bytes/1024:.1f} KB')
print(f'FP32 equivalent: {tensor_info.n_elements*4/1024:.1f} KB')
print(f'Compression: {tensor_info.n_elements*4/tensor_info.size_bytes:.2f}x')
print()

# Load and dequantize
print('Loading and dequantizing...')
tensor = reader.load_tensor(tensor_name)
print(f'Dequantized shape: {tensor.shape}, dtype: {tensor.dtype}')
print(f'Value range: [{tensor.min():.4f}, {tensor.max():.4f}]')
print(f'Mean: {tensor.mean():.4f}, Std: {tensor.std():.4f}')

In [ ]:
# Part 5: Visualize Q4_K block structure
# Show how scales vary across blocks for one layer

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Weight distribution
axes[0].hist(tensor.flatten().numpy(), bins=100, color='steelblue', alpha=0.8)
axes[0].set_title(f'{tensor_name}\nWeight distribution after Q4_K dequantization')
axes[0].set_xlabel('Weight value')
axes[0].set_ylabel('Count')
axes[0].axvline(x=0, color='red', linewidth=1, linestyle='--')

# Reshape to show rows/columns pattern
if tensor.dim() == 2:
    W = tensor[:32, :64]  # show a 32x64 slice
    im = axes[1].imshow(W.numpy(), cmap='RdBu', aspect='auto')
    axes[1].set_title(f'Weight heatmap\n(32x64 slice of {tensor.shape[0]}x{tensor.shape[1]})')
    axes[1].set_xlabel('Input dimension')
    axes[1].set_ylabel('Output dimension')
    plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Part 6: Manually decode one Q8_0 block to understand the binary format
print('=== Manual Q8_0 Block Decoding ===')
print('This shows exactly what is stored on disk and how we decode it.')
print()

# Create a synthetic Q8_0 block with known values
d = 0.05    # scale = 0.05
qs = list(range(-16, 16))   # 32 int8 values from -16 to 15

# Build the raw bytes (as stored in GGUF)
raw = struct.pack('<e', d) + struct.pack('<32b', *qs)
print(f'Raw bytes (hex): {raw[:8].hex()} ... ({len(raw)} total bytes)')
print(f'Breakdown:')
print(f'  Bytes 0-1: {raw[0:2].hex()} = scale d = {struct.unpack_from("<e", raw, 0)[0]}')
print(f'  Bytes 2-5: {raw[2:6].hex()} = int8 values {struct.unpack_from("<4b", raw, 2)}')
print()

# Dequantize
result = dequantize_q8_0(raw, (32,))
print('Dequantized values:')
print(f'  {result.tolist()}')
print()
print(f'Expected (d * q): {[d * q for q in qs][:5]}...')
print(f'Match: {torch.allclose(result, torch.tensor([d * q for q in qs]))}')

In [ ]:
# Part 7: Compare GGUF Q4_K_M vs our absmax INT8 on the same layer
from src.quantization.absmax import absmax_quantize, absmax_dequantize

# GGUF version (already loaded above)
W_gguf = tensor.float()

# Our absmax INT8 applied to the GGUF tensor
q_our, s_our = absmax_quantize(W_gguf)
W_our = absmax_dequantize(q_our, s_our)

# Compare
diff = (W_gguf - W_our).abs()
print(f'Comparison: GGUF Q4_K_M vs our absmax INT8')
print(f'  Mean diff: {diff.mean():.6f}')
print(f'  Max diff:  {diff.max():.6f}')
print()
print('(This diff reflects that Q4_K_M and INT8 are different quantization schemes')
print(' applied to the same original model — the GGUF weights are already dequantized')
print(' approximations, and we are then re-quantizing them with INT8)')

## Summary

Key takeaways:

1. **GGUF is a self-contained binary format** — one file has weights, metadata, and tokenizer
2. **Tensor data is back-to-back with no gaps** (aligned to 32 bytes between info and data)
3. **Q4_K_M stores 256 weights in 144 bytes** (~4.5 bits/weight including scale overhead)
4. **Reading is lazy** — our reader only loads what you ask for with `load_tensor(name)`
5. **The GGUF-to-HF name map** in `loader.py` bridges llama.cpp naming to HuggingFace naming

**Next:** `04_benchmark_dashboard.ipynb` — run all schemes and visualize the tradeoffs